# Análises de Negócio

## Objetivo

Utilizar a camada Ouro para analisar as avaliações de filmes provenientes
do IMDb e do Rotten Tomatoes e demonstrar como a arquitetura de dados
construída suporta consultas analíticas.

## Perguntas analíticas

1. Qual é a distribuição das avaliações positivas e negativas em cada fonte?
2. As fontes apresentam diferenças na proporção de sentimentos?
3. Como as avaliações do Rotten Tomatoes se distribuem entre críticos
   Top Critic e demais críticos?
4. Como evolui temporalmente o sentimento das avaliações do
   Rotten Tomatoes?
5. Quais filmes concentram maior quantidade de avaliações no
   Rotten Tomatoes?
6. Entre filmes com quantidade relevante de avaliações, quais apresentam
   maiores e menores proporções de avaliações positivas?

## Premissa de interpretação

IMDb e Rotten Tomatoes possuem naturezas distintas.

No IMDb, o conjunto utilizado contém textos classificados como positivos
ou negativos. No Rotten Tomatoes, a classificação foi harmonizada a partir
de Fresh/Rotten.

Assim, a comparação entre as fontes será descritiva e não será interpretada
como equivalência metodológica entre os dois conjuntos de dados.

In [0]:
# ============================================================
# Carregamento e contrato de entrada
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABELA_FATO = "mvp3_eng_dados.ouro.fato_avaliacao"
TABELA_DIM_FILME = "mvp3_eng_dados.ouro.dim_filme"
TABELA_DIM_DATA = "mvp3_eng_dados.ouro.dim_data"

df_fato = spark.table(TABELA_FATO)
df_dim_filme = spark.table(TABELA_DIM_FILME)
df_dim_data = spark.table(TABELA_DIM_DATA)

colunas_necessarias_fato = {
    "id_avaliacao_global",
    "id_fonte",
    "id_filme",
    "id_data",
    "sentimento",
    "tipo_avaliador",
    "eh_top_critic"
}

ausentes = colunas_necessarias_fato - set(df_fato.columns)

print(f"Registros disponíveis: {df_fato.count():,}")
print(f"Colunas ausentes: {sorted(ausentes)}")

assert len(ausentes) == 0, (
    f"Contrato de entrada inválido. "
    f"Colunas ausentes: {sorted(ausentes)}"
)

print("✓ Contrato analítico validado.")

Registros disponíveis: 1,060,128
Colunas ausentes: []
✓ Contrato analítico validado.


In [0]:
# ============================================================
# Respondendo as perguntas 1 e 2
# Distribuição de sentimentos por fonte
# ============================================================

df_sentimento_fonte = (
    df_fato
    .join(
        df_dim_fonte := spark.table(
            "mvp3_eng_dados.ouro.dim_fonte"
        ),
        on="id_fonte",
        how="inner"
    )
    .groupBy("fonte", "sentimento")
    .agg(
        F.count("*").alias("quantidade")
    )
)

janela_fonte = Window.partitionBy("fonte")

df_sentimento_fonte = (
    df_sentimento_fonte
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.sum("quantidade").over(janela_fonte)
            * 100,
            2
        )
    )
    .orderBy("fonte", "sentimento")
)

display(df_sentimento_fonte)

fonte,sentimento,quantidade,percentual
IMDb,NEGATIVO,24698,49.81
IMDb,POSITIVO,24884,50.19
Rotten Tomatoes,NEGATIVO,366614,36.28
Rotten Tomatoes,POSITIVO,643932,63.72


In [0]:
# ============================================================
# Rotten Tomatoes — Top Critic × demais críticos
# ============================================================

df_rt_criticos = (
    df_fato
    .join(
        df_dim_fonte.select("id_fonte", "fonte"),
        on="id_fonte",
        how="inner"
    )
    .filter(F.col("fonte") == "Rotten Tomatoes")
    .filter(F.col("eh_top_critic").isNotNull())
    .withColumn(
        "categoria_critico",
        F.when(
            F.col("eh_top_critic") == True,
            F.lit("TOP CRITIC")
        ).otherwise(
            F.lit("DEMAIS CRÍTICOS")
        )
    )
    .groupBy("categoria_critico", "sentimento")
    .agg(
        F.count("*").alias("quantidade")
    )
)

janela_critico = Window.partitionBy("categoria_critico")

df_rt_criticos = (
    df_rt_criticos
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.sum("quantidade").over(janela_critico)
            * 100,
            2
        )
    )
    .orderBy("categoria_critico", "sentimento")
)

display(df_rt_criticos)

categoria_critico,sentimento,quantidade,percentual
DEMAIS CRÍTICOS,NEGATIVO,260827,34.68
DEMAIS CRÍTICOS,POSITIVO,491348,65.32
TOP CRITIC,NEGATIVO,105787,40.94
TOP CRITIC,POSITIVO,152584,59.06


In [0]:
# ============================================================
# Evolução temporal do sentimento — Rotten Tomatoes
# ============================================================

df_evolucao_rt = (
    df_fato
    .join(
        df_dim_fonte.select("id_fonte", "fonte"),
        on="id_fonte",
        how="inner"
    )
    .join(
        df_dim_data.select("id_data", "ano"),
        on="id_data",
        how="inner"
    )
    .filter(F.col("fonte") == "Rotten Tomatoes")
    .groupBy("ano", "sentimento")
    .agg(
        F.count("*").alias("quantidade")
    )
)

janela_ano = Window.partitionBy("ano")

df_evolucao_rt = (
    df_evolucao_rt
    .withColumn(
        "percentual",
        F.round(
            F.col("quantidade")
            / F.sum("quantidade").over(janela_ano)
            * 100,
            2
        )
    )
    .orderBy("ano", "sentimento")
)

display(df_evolucao_rt)

ano,sentimento,quantidade,percentual
1800,NEGATIVO,28,37.33
1800,POSITIVO,47,62.67
1923,NEGATIVO,1,100.0
1927,POSITIVO,2,100.0
1928,POSITIVO,1,100.0
1929,NEGATIVO,1,100.0
1931,NEGATIVO,1,100.0
1932,POSITIVO,1,100.0
1933,POSITIVO,1,100.0
1934,POSITIVO,1,100.0


In [0]:
# ============================================================
# Filmes com maior número de avaliações
# ============================================================

df_ranking_volume = (
    df_fato
    .filter(F.col("id_filme").isNotNull())
    .join(
        df_dim_filme.select(
            "id_filme",
            "titulo_filme"
        ),
        on="id_filme",
        how="inner"
    )
    .groupBy(
        "id_filme",
        "titulo_filme"
    )
    .agg(
        F.count("*").alias("quantidade_avaliacoes")
    )
    .orderBy(
        F.desc("quantidade_avaliacoes"),
        F.asc("titulo_filme")
    )
)

display(df_ranking_volume.limit(20))

id_filme,titulo_filme,quantidade_avaliacoes
1614ee301d8fea9d7f45bb8530f2888182498420647ed1705065bf05301af721,Joker,574
952ad5e41f46c95f230200b95e26ce0a63a3e7c8d2c8b07f9356ac8e532df12e,Once Upon a Time In Hollywood,554
34bc20bb5aa3c6302f9c7494ddc64cc1e196f3607d2865572fc764b8708b6347,Us,535
6b07d3a74fb9ed3f93d9dade9cf3c22224aa25f33b932983157e51cd9abb6ad5,Avengers: Endgame,528
43af4754116ac09940e0866fdebaa860cb493243e3868f8c0ecbf308947472c0,Captain Marvel,523
8e8d0b6f1b367110e4716a21da085722c2ffe37bccfbddc233263e05dc7c64de,A Star Is Born,517
22f84f90a0cb5d0c0d7bb6a2091a21d52264020afaa36546ff15f4ad77eb7a81,Black Panther,512
8b6206df1d69bc5ab3994f5c33a58f737b9fb8e85de12bace2019f7b3825b388,Star Wars: The Rise of Skywalker,496
b6cd77c6eaea3d78ca079d8bd3b9491820605d4a391cd2ebe428afdb873a5779,Solo: A Star Wars Story,474
12c6c69eb623a38303272a397add70eaa04d529757343aa6fe12285b7135c17d,Star Wars: The Last Jedi,473


In [0]:
# ============================================================
# Distribuição do volume de avaliações por filme
# ============================================================

df_distribuicao_volume = (
    df_ranking_volume
    .agg(
        F.count("*").alias("quantidade_filmes"),

        F.min("quantidade_avaliacoes")
        .alias("minimo"),

        F.expr(
            "percentile_approx(quantidade_avaliacoes, 0.25)"
        ).alias("percentil_25"),

        F.expr(
            "percentile_approx(quantidade_avaliacoes, 0.50)"
        ).alias("mediana"),

        F.expr(
            "percentile_approx(quantidade_avaliacoes, 0.75)"
        ).alias("percentil_75"),

        F.expr(
            "percentile_approx(quantidade_avaliacoes, 0.90)"
        ).alias("percentil_90"),

        F.max("quantidade_avaliacoes")
        .alias("maximo")
    )
)

display(df_distribuicao_volume)

quantidade_filmes,minimo,percentil_25,mediana,percentil_75,percentil_90,maximo
17706,1,12,28,75,154,574


In [0]:
# ============================================================
# Positividade por filme
# Critério: filmes com pelo menos 75 avaliações
# (percentil 75 da distribuição observada)
# ============================================================

LIMIAR_AVALIACOES = 75

df_positividade_filme = (
    df_fato
    .filter(F.col("id_filme").isNotNull())
    .join(
        df_dim_filme.select(
            "id_filme",
            "titulo_filme"
        ),
        on="id_filme",
        how="inner"
    )
    .groupBy(
        "id_filme",
        "titulo_filme"
    )
    .agg(
        F.count("*").alias("quantidade_avaliacoes"),

        F.sum(
            F.when(
                F.col("sentimento") == "POSITIVO",
                1
            ).otherwise(0)
        ).alias("avaliacoes_positivas"),

        F.sum(
            F.when(
                F.col("sentimento") == "NEGATIVO",
                1
            ).otherwise(0)
        ).alias("avaliacoes_negativas")
    )
    .filter(
        F.col("quantidade_avaliacoes") >= LIMIAR_AVALIACOES
    )
    .withColumn(
        "percentual_positivo",
        F.round(
            F.col("avaliacoes_positivas")
            / F.col("quantidade_avaliacoes")
            * 100,
            2
        )
    )
)

quantidade_filmes_analisados = df_positividade_filme.count()

print(
    f"Limiar mínimo: {LIMIAR_AVALIACOES} avaliações"
)
print(
    f"Filmes incluídos na análise: "
    f"{quantidade_filmes_analisados:,}"
)

print("\nFILMES COM MAIOR PROPORÇÃO POSITIVA")

display(
    df_positividade_filme
    .orderBy(
        F.desc("percentual_positivo"),
        F.desc("quantidade_avaliacoes"),
        F.asc("titulo_filme")
    )
    .limit(20)
)

print("\nFILMES COM MENOR PROPORÇÃO POSITIVA")

display(
    df_positividade_filme
    .orderBy(
        F.asc("percentual_positivo"),
        F.desc("quantidade_avaliacoes"),
        F.asc("titulo_filme")
    )
    .limit(20)
)

Limiar mínimo: 75 avaliações
Filmes incluídos na análise: 4,469

FILMES COM MAIOR PROPORÇÃO POSITIVA


id_filme,titulo_filme,quantidade_avaliacoes,avaliacoes_positivas,avaliacoes_negativas,percentual_positivo
b75488331023d25ecb24acc6012277db308db6844c09372d0ff61522408f8ee5,Paddington 2,242,242,0,100.0
1af1814f0e6043b855db0ee393f839058fc01cbb2e9416849961a9c870aca1f5,Leave No Trace,234,234,0,100.0
30a768aaaf642638ef13ac8d84988096ebffc1a748ada984fd63f3d4b11bf213,Toy Story 2,169,169,0,100.0
22f7e5855e3676bc473badf376d4949d3e87696e92666c6e9749ccbe184a7358,Man on Wire,158,158,0,100.0
be11532e519db494dd8c4f847c06945578990477c93356776a62e4456cb62c49,Taxi to the Dark Side,93,93,0,100.0
ac529ecbd487f29638bf5845350eadf3c16ae4bf7ce7b16daf2d1bcc3732e220,Citizen Kane,92,92,0,100.0
0d84260599cd4fac6b2064d4d1223ff2ef5183564881cf9118511c264a796078,Toy Story,90,90,0,100.0
b2b8b71b19d11427c3e2b067272c4de3d2888733f860f0f4582ca9e53246e498,Crip Camp: A Disability Revolution,84,84,0,100.0
eaf0954fdf52365e591b4ef77f566ad85c98fe53ad2e3c92530c7ea0ad599f31,GETT: The Trial of Viviane Amsalem,78,78,0,100.0
8a038440c94a53d0d032ac4aed7cb39c6d06c3edc6f9441ee1a405f8a46ffd4d,A Separation,176,175,1,99.43



FILMES COM MENOR PROPORÇÃO POSITIVA


id_filme,titulo_filme,quantidade_avaliacoes,avaliacoes_positivas,avaliacoes_negativas,percentual_positivo
b0ff5787440505c4142a01d60b4c258e6bb0ad61b10827de601c5ba5615cc79f,Ballistic: Ecks vs. Sever,118,0,118,0.0
0c02ffe11b74a51ca47486fc203e5d89c40888b794f775d18c6febcf18fac43a,One Missed Call,81,0,81,0.0
3dc4db926016fc778ccfe776dac77bbe31db004155f441b985f6f31ba64f3efd,Alone in the Dark,123,1,122,0.81
9698c85b129ad1e49fa7c448c5d248805347586b94cebc64db4220a70865f3af,The Master of Disguise,104,1,103,0.96
a1a26a4d4a366c53d386be7e5797e015491db8cc5cd1928c2b9b40675475023b,Daddy Day Camp,81,1,80,1.23
f8df0a0e2472864beeacbf48a7dc9458edfec96e4ee49c933523bafb75e98095,Twisted,135,2,133,1.48
a76d4ef68d1605fbeca966f6bace4cf0bb4a444a9493a118736aaef5cc659ceb,Jack and Jill,116,3,113,2.59
37cfe56b0d36f84dbb6933189108c64068ffe159048b699dff40c02911f4d35f,Battlefield Earth,152,4,148,2.63
44b4eb7bb047a8ce58e12c993f3fd2aaa9cc84f1424f82b6beda287db1d73bc4,Deuces Wild,75,2,73,2.67
b91522c18e7dc4b92ea23907ba8c83875a785700c7a0b89435fd608727312def,Bless the Child,111,3,108,2.7


In [0]:
# ============================================================
# Síntese quantitativa das análises
# ============================================================

total_avaliacoes = df_fato.count()

total_fontes = (
    df_fato
    .select("id_fonte")
    .distinct()
    .count()
)

total_filmes = (
    df_fato
    .filter(F.col("id_filme").isNotNull())
    .select("id_filme")
    .distinct()
    .count()
)

total_datas = (
    df_fato
    .filter(F.col("id_data").isNotNull())
    .select("id_data")
    .distinct()
    .count()
)

total_positivas = (
    df_fato
    .filter(F.col("sentimento") == "POSITIVO")
    .count()
)

total_negativas = (
    df_fato
    .filter(F.col("sentimento") == "NEGATIVO")
    .count()
)

percentual_positivo = (
    total_positivas / total_avaliacoes * 100
)

percentual_negativo = (
    total_negativas / total_avaliacoes * 100
)

print("=" * 68)
print("SÍNTESE DAS ANÁLISES DE NEGÓCIO")
print("=" * 68)

print("\nVOLUMETRIA")
print(f"Avaliações analisadas: {total_avaliacoes:,}")
print(f"Fontes:                {total_fontes:,}")
print(f"Filmes identificados:  {total_filmes:,}")
print(f"Datas identificadas:   {total_datas:,}")

print("\nSENTIMENTO GLOBAL")
print(
    f"Positivas: {total_positivas:,} "
    f"({percentual_positivo:.2f}%)"
)
print(
    f"Negativas: {total_negativas:,} "
    f"({percentual_negativo:.2f}%)"
)

print("\nANÁLISE POR FILME")
print(
    f"Limiar adotado: {LIMIAR_AVALIACOES} avaliações"
)
print(
    f"Filmes acima do limiar: "
    f"{quantidade_filmes_analisados:,}"
)

print("\nOBSERVAÇÕES METODOLÓGICAS")
print("✓ IMDb e Rotten Tomatoes analisados separadamente por fonte")
print("✓ sentimentos harmonizados em POSITIVO e NEGATIVO")
print("✓ análise temporal restrita aos registros com data")
print("✓ ranking de filmes restrito aos registros com filme associado")
print(
    "✓ comparação de positividade utiliza limiar baseado "
    "no percentil 75"
)

print("=" * 68)

SÍNTESE DAS ANÁLISES DE NEGÓCIO

VOLUMETRIA
Avaliações analisadas: 1,060,128
Fontes:                2
Filmes identificados:  17,706
Datas identificadas:   8,015

SENTIMENTO GLOBAL
Positivas: 668,816 (63.09%)
Negativas: 391,312 (36.91%)

ANÁLISE POR FILME
Limiar adotado: 75 avaliações
Filmes acima do limiar: 4,469

OBSERVAÇÕES METODOLÓGICAS
✓ IMDb e Rotten Tomatoes analisados separadamente por fonte
✓ sentimentos harmonizados em POSITIVO e NEGATIVO
✓ análise temporal restrita aos registros com data
✓ ranking de filmes restrito aos registros com filme associado
✓ comparação de positividade utiliza limiar baseado no percentil 75


# Conclusões das análises

## 1. Integração das fontes

A camada Ouro permitiu consultar de forma integrada avaliações provenientes
do IMDb e do Rotten Tomatoes. As classificações originais foram preservadas
nas etapas anteriores do pipeline e harmonizadas semanticamente para as
categorias POSITIVO e NEGATIVO.

A harmonização possibilita análises conjuntas, mas não implica equivalência
metodológica entre os datasets. IMDb e Rotten Tomatoes possuem origens,
estruturas e processos de classificação distintos.

## 2. Comparação de sentimentos

A distribuição entre avaliações positivas e negativas foi analisada
separadamente por fonte. Essa abordagem é mais adequada que a utilização
isolada da distribuição global, pois os datasets possuem volumes de
registros significativamente diferentes.

## 3. Perfil dos críticos

Para o Rotten Tomatoes, a disponibilidade do atributo `eh_top_critic`
permitiu comparar descritivamente a distribuição dos sentimentos entre
Top Critics e os demais críticos.

## 4. Análise temporal

A análise temporal foi realizada apenas para registros que possuem data
de avaliação. O dataset IMDb utilizado neste projeto não fornece a data
necessária para essa análise, enquanto o Rotten Tomatoes possui informação
temporal disponível.

A ausência de data nos registros do IMDb foi preservada como característica
do dado de origem, evitando a criação artificial de informações.

## 5. Análise por filme

A associação com a dimensão de filmes permitiu identificar os títulos com
maior quantidade de avaliações e analisar a proporção de sentimentos por
filme.

Para reduzir distorções provocadas por filmes com poucas avaliações, a
comparação de positividade foi limitada aos filmes com pelo menos 75
avaliações. Esse valor corresponde ao percentil 75 da distribuição observada
de quantidade de avaliações por filme.

## 6. Considerações finais

As análises demonstram que a arquitetura construída permite partir dos
arquivos originais, preservar sua rastreabilidade, tratar problemas de
qualidade, harmonizar fontes heterogêneas e disponibilizar dados estruturados
para consultas analíticas na camada Ouro.

Os resultados devem ser interpretados dentro do universo dos datasets
utilizados e não como representação de todas as avaliações existentes sobre
os filmes.